# 01. PubChem 기반 데이터 수집

이 실습에서는 **PubChem REST API (PUG-REST)** 를 이용해 URL 요청만으로
화합물(Chemical) 정보와 생리활성(BioAssay) 데이터를 직접 수집한다.

## PUG-REST 란?
PubChem의 **P**ower **U**ser **G**ateway - **REST** 인터페이스로,
웹 주소(URL)를 조합해 GET 요청을 보내면 JSON/CSV/PNG 형태로 데이터를 돌려준다.
별도의 라이브러리 없이 `requests` 만으로 사용할 수 있다는 것이 장점이다.

```
https://pubchem.ncbi.nlm.nih.gov/rest/pug / <입력> / <조회 대상> / <출력형식>
                     (base)                  (input)   (operation)    (output)
```

- 공식 문서: https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest

## 0. 환경 설정

실습에 필요한 라이브러리를 불러온다. (Google Colab에는 모두 기본 설치되어 있음)
- `requests` : REST API에 HTTP 요청을 보내는 라이브러리
- `pandas`   : 수집한 데이터를 표(DataFrame) 형태로 다루기 위한 라이브러리
- `time`     : PubChem 서버 부하 방지를 위한 요청 간 대기

In [12]:
import requests           # REST API(URL) 요청
import pandas as pd       # 데이터프레임 처리
import time               # 요청 간 간격 조절
from pprint import pprint

---
# 1. PubChem Chemical(Compound) 데이터 수집

> **[실습] REST API 기반 (url) PubChem Chemical 데이터 수집**

PubChem의 화합물 페이지(예: [Aspirin, CID 2244](https://pubchem.ncbi.nlm.nih.gov/compound/2244))
에 정리된 정보(분자식, 분자량, 별명, 구조, 설명 등)를 API로 그대로 가져와 본다.

### Compound 요청 URL 구조
```
{BASE}/compound/{입력방식}/{값}/{조회대상}/{출력형식}
```
| 구성 | 예시 | 설명 |
|------|------|------|
| 입력방식(input) | `name`, `cid`, `smiles` | 무엇으로 찾을지 |
| 값 | `aspirin`, `2244` | 검색 값 |
| 조회대상(operation) | `cids`, `property/...`, `synonyms`, `description` | 무엇을 가져올지 |
| 출력형식(output) | `JSON`, `CSV`, `PNG`, `TXT` | 어떤 형식으로 받을지 |

## 1-1. 화합물 이름 → CID 조회

CID(Compound ID)는 PubChem이 화합물마다 부여하는 고유 번호다.
먼저 화합물 **이름(name)** 으로 CID를 찾아본다. (Aspirin → 2244)

In [13]:
# PUG-REST 서비스의 기본 주소(base URL). 이후 모든 요청은 이 주소 뒤에 경로를 붙여 만든다.
# 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'

# REST의 규칙에 따라 asiprin에 대해서 'name' 으로 'cid'를 조회하는 url 주소
# name → CID : 기본주소/compound/name/{이름}/cids/JSON
url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/cids/JSON'

# url 주소에 대해서 GET 요청 전송
response = requests.get(url, verify=False)

# 요청에 대한 응답(JSON 파일형태)을 파이썬 dict로 변환 및 받아오기
data = response.json()  

pprint(data)

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'pubchem.ncbi.nlm.nih.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'IdentifierList': {'CID': [2244]}}


In [10]:
# cid의 정보는 'IdentifierList' - 'CID' - 첫번째 값
cid = data['IdentifierList']['CID'][0]
print('Asiprin 의 CID =', cid)

Asiprin 의 CID = 2244


## 1-2. 화합물 속성(Property) 수집

슬라이드의 **Molecular Formula / Molecular Weight** 등에 해당하는 값들을 가져온다.
여러 속성을 쉼표로 이어서 한 번에 요청할 수 있다.

- `MolecularFormula` : 분자식 (예: C9H8O4)
- `MolecularWeight`  : 분자량 (예: 180.16 g/mol)
- `IUPACName`        : IUPAC 명명법 이름
- `SMILES`           : 구조를 나타내는 문자열(SMILES)

In [ ]:
# 가져올 속성(Property) 목록
# MolecularFormula / MolecularWeight / IUPACName / SMILES

# REST의 규칙에 따라 asiprin에 대해서 'cid' 으로 'Property'를 조회하는 url 주소
# cid → property : 기본주소/compound/cid/{CID}/property/{속성1,속성2,속성3,...}/JSON
url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/property/MolecularFormula,MolecularWeight,IUPACName,SMILES/JSON'

# url 주소에 대해서 GET 요청 전송
response = requests.get(url, verify=False)

# 요청에 대한 응답(JSON 파일형태)을 파이썬 dict로 변환 및 받아오기
data = response.json()  
pprint(data)


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'pubchem.ncbi.nlm.nih.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'PropertyTable': {'Properties': [{'CID': 2244,
                                   'IUPACName': '2-acetyloxybenzoic acid',
                                   'MolecularFormula': 'C9H8O4',
                                   'MolecularWeight': '180.16',
                                   'SMILES': 'CC(=O)OC1=CC=CC=C1C(=O)O'}]}}


In [ ]:
# properties의 정보는 'PropertyTable' - 'Properties' - 첫번째 값
properties = data['PropertyTable']['Properties'][0]
pprint(properties)

{'CID': 2244,
 'IUPACName': '2-acetyloxybenzoic acid',
 'MolecularFormula': 'C9H8O4',
 'MolecularWeight': '180.16',
 'SMILES': 'CC(=O)OC1=CC=CC=C1C(=O)O'}


In [ ]:
print('Asiprin의 MF:', properties['MolecularFormula'],
      ', MolecualrWeight:', properties['MolecularWeight'],
      ', IUPAC Name:', properties['IUPACName'],
      ', SMILES:', properties['SMILES'])

# properties(dict)를 pandas DataFrame(표)으로 변환
#   - 딕셔너리(dict)를 리스트로 감싸면([ ... ]) 한 줄짜리 표가 만들어진다.
df_property = pd.DataFrame([properties])
df_property

Asiprin의 MF: C9H8O4 , MolecualrWeight: 180.16 , IUPAC Name: 2-acetyloxybenzoic acid , SMILES: CC(=O)OC1=CC=CC=C1C(=O)O


,CID,MolecularFormula,MolecularWeight,SMILES,IUPACName
0,2244,C9H8O4,180.16,CC(=O)OC1=CC=CC=C1C(=O)O,2-acetyloxybenzoic acid


## 1-3. [실습] 이름(name) → 속성(Property) 직접 조회하기

> **직접 해보기 🖐️**

앞의 **1-1**에서는 이름으로 CID를 먼저 찾고, **1-2**에서 속성을 가져왔다.
이번에는 CID를 거치지 않고 **화합물 이름만으로 곧바로 속성**을 조회해 본다.

### 요청 URL 구조
```
{BASE}/compound/name/{이름}/property/{속성1,속성2,...}/JSON
```

**미션** — 아래 코드 셀의 빈칸(`______`)을 채워 Aspirin의 속성을 출력해 보자.
| 가져올 속성 | 의미 |
|------|------|
| `MolecularFormula` | 분자식 |
| `MolecularWeight`  | 분자량 |
| `IUPACName`        | IUPAC 이름 |
| `SMILES`           | 구조 문자열 |

> 💡 막히면 **1-2 셀의 코드**를 그대로 참고하면 된다.

In [ ]:
# =====================================================================
# [실습] 화합물 '이름(name)'으로 곧바로 속성(Property) 조회하기
#   - 위 1-2 셀은 name → property 를 이미 사용했다. 직접 처음부터 작성해 보자.
#   - 아래 밑줄(______) 부분을 채우면 된다. (막히면 위의 코드 참조!)
# =====================================================================

# [1] 요청 URL 만들기
#     구조 : 기본주소/compound/name/{이름}/property/{속성1,속성2,...}/JSON
#     조회할 속성 : MolecularFormula, MolecularWeight, IUPACName, SMILES
url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/______/property/______/JSON'

# [2] GET 요청 보내기
response = requests.get(url, verify=False)

# [3] 응답(JSON)을 파이썬 dict 로 변환
data = response.json()

# [4] properties 정보만 꺼내기 : 'PropertyTable' → 'Properties' → 첫 번째 값
properties = data['______']['______'][0]

# [5] 결과 출력
pprint(properties)
df_property = pd.DataFrame([properties])
df_property

---
# 2. PubChem BioAssay 데이터 수집

> **[실습] REST API 기반 (url) PubChem BioAssay 데이터 수집**
>
> PubChem REST API BioAssay 로부터 chemical 및 활성 데이터 수집

**BioAssay** 는 화합물의 생리활성 실험 결과를 모아 둔 데이터로,
각 실험은 **AID(Assay ID)** 라는 고유 번호를 가진다.

이번 실습 대상은 슬라이드와 동일하게
[**AID 651580** — *Single concentration confirmation of uHTS identification of HIF-2a Inhibitors*](https://pubchem.ncbi.nlm.nih.gov/bioassay/651580)
이다. (Target: HIF-2a / endothelial PAS domain-containing protein 1)

### Assay 요청 URL 구조
```
{BASE}/assay/aid/{AID}/{조회대상}/{출력형식}
```
- `description` : 어세이 개요(제목, 타겟, 출처 등)
- `cids`        : 실험된 화합물 CID 목록 (전체/Active/Inactive)
- `CSV`         : 화합물별 활성 결과 데이터 표

## 2-1. Assay 개요(Description) 수집

슬라이드의 **Protein Target / Source / BioAssay Type / Description** 등
어세이의 기본 정보를 가져온다.

In [29]:
# PUG-REST 서비스의 기본 주소(base URL). 이후 모든 요청은 이 주소 뒤에 경로를 붙여 만든다.
# 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'
# 실습 대상 BioAssay ID (HIF-2a 저해제)
# AID = 651580   

# REST의 규칙에 따라 특정 BioAssay(aid = 651580)에 대해서 'AID' 으로 'description'를 조회하는 url 주소
# aid → description : 기본주소/assay/aid/{AID}/description/JSON
url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/651580/description/JSON'

# url 주소에 대해서 GET 요청 전송
response = requests.get(url, verify=False)

# 요청에 대한 응답(JSON 파일형태)을 파이썬 dict로 변환 및 받아오기
data = response.json()  
pprint(data)

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'pubchem.ncbi.nlm.nih.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'PC_AssayContainer': [{'assay': {'descr': {'activity_outcome_method': 1,
                                            'aid': {'id': 651580, 'version': 1},
                                            'aid_source': {'db': {'name': 'Burnham '
                                                                          'Center '
                                                                          'for '
                                                                          'Chemical '
                                                                          'Genomics',
                                                                  'source_id': {'str': 'SBCCG-A900-HIF-2a-CP-Assay'}}},
                                            'comment': ['Compounds were tested '
                                                        'in quadruplicates and '
                                                        'those that have % '
                                                        'activit

In [ ]:
# Assay의 정보는 'PC_AssayContainer' - 첫 번째값 - 'assay' - ['descr]
assay = data['PC_AssayContainer'][0]['assay']['descr']

print('AID   :', assay['aid']['id'])
print('제목  :', assay['name'])
# 어세이 설명은 여러 줄의 리스트로 제공된다
print('설명  :')
for line in assay.get('description', [])[:5]:
    print('  ', line)

AID   : 651580
제목  : Single concentration confirmation of uHTS identification of HIF-2a Inhibitors in a luminesence assay
설명  :
   Data Source: Sanford-Burnham Center for Chemical Genomics (SBCCG)
   Source Affiliation: Sanford-Burnham Medical Research Institute (SBMRI, La Jolla, CA)
   Network: NIH Molecular Libraries Probe Production Centers Network (MLPCN)
   Grant Number: 1 R03 DA033980-01A1
   Assay Provider: Mei Yee Koh, Ph.D.,  University of Texas M.D. Anderson Cancer Center


## 2-2. 화합물별 활성 데이터 표(CSV) 수집

출력형식을 `CSV` 로 지정하면 화합물별 활성 결과를 표로 받을 수 있다.
핵심 컬럼은 다음과 같다.
- `PUBCHEM_CID`               : 화합물 ID
- `PUBCHEM_ACTIVITY_OUTCOME`  : 활성 판정 (Active / Inactive)
- `PUBCHEM_ACTIVITY_SCORE`    : 활성 점수

슬라이드의 **Tested Substances: All(1,759) / Active(982) / Inactive(777)** 에 해당하는
원본 데이터다.

In [45]:
from io import StringIO
# PUG-REST 서비스의 기본 주소(base URL). 이후 모든 요청은 이 주소 뒤에 경로를 붙여 만든다.
# 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'
# 실습 대상 BioAssay ID (HIF-2a 저해제)
# AID = 651580   

# REST의 규칙에 따라 특정 BioAssay(aid = 651580)에 대해서 'AID' 으로 'description'를 조회하는 url 주소
# aid → description : 기본주소/assay/aid/{AID}/description/JSON
url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/651580/CSV'

# url 주소에 대해서 GET 요청 전송
response = requests.get(url, verify=False)

# 요청에 대한 응답(CSV)을 pandas 표(DataFrame)로 바로 받아오기
data = StringIO(response.text)
data = pd.read_csv(data)
display(data)


# data = pd.read_csv(url)
# data

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'pubchem.ncbi.nlm.nih.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,PUBCHEM_ACTIVITY_URL,PUBCHEM_ASSAYDATA_COMMENT,%Activity at 18.75 uM_Mean,%Activity at 18.75 uM_1,...,%Activity at 18.75 uM_3,%Activity at 18.75 uM_4,Value_1,Value_2,Value_3,Value_4,Mean High,STD Deviation High,Mean Low,STD Deviation Low
0,RESULT_TYPE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FLOAT,FLOAT,...,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT
1,RESULT_DESCR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,% inhibition for the first replicate,...,% inhibition for the third replicate,% inhibition for the fourth replicate,Value of the sample,NaN,NaN,NaN,Mean luminesence of negative controls in the c...,Standard deviation (n=64) of negative controls...,Mean luminesence of positive controls in the c...,Standard deviation (n=64) of positive controls...
2,RESULT_UNIT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PERCENT,PERCENT,...,PERCENT,PERCENT,RLU,RLU,RLU,RLU,RLU,RLU,RLU,RLU
3,RESULT_ATTR_CONC_MICROMOL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.75,18.75,...,18.75,18.75,18.75,18.75,18.75,18.75,NaN,NaN,NaN,NaN
4,1,85146295.0,44142181.0,C=CS(=O)(=O)N(CC1=CC=CC=C1Br)[C@@H](CC2=CC=CC=...,Inactive,21.0,NaN,NaN,35.14,34.86,...,33.44,36.18,579.04,568.21,591.53,567.48,0,9.38,100,0.43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1758,1755,24836171.0,4882569.0,CC1=CC(=C(C=C1)NC(=O)C(=CC2=CC=C(C=C2)C(=O)O)C...,Active,30.0,NaN,NaN,92.03,92,...,91.54,91.19,78.87,67.03,82.75,85.72,0,9.73,100,1.01
1759,1756,57261817.0,2482686.0,C1=CC=C(C=C1)CNC(=O)COC(=O)C2=NC(=C(C(=C2Cl)Cl...,Inactive,21.0,NaN,NaN,17.66,22.38,...,12.42,13.74,668.64,671.19,753.05,741.83,0,9.73,100,1.01
1760,1757,17407664.0,2874907.0,CC1=CC(NC2=C1C=C(C=C2)OC(=O)C(C)N3C(=O)C4=CC=C...,Active,30.0,NaN,NaN,89.67,89.16,...,90.18,90.21,102.92,103.14,94.29,94,0,9.73,100,1.01
1761,1758,57256244.0,16302071.0,COC1=CC=CC(=C1)CNC(=O)COC(=O)C2=NC(=C(C(=C2Cl)...,Inactive,21.0,NaN,NaN,31.96,30.64,...,32.61,35.02,598.68,607.84,582.03,561.56,0,9.73,100,1.01


In [46]:
df_assay = data.dropna(subset=['PUBCHEM_CID']).reset_index(drop=True)
df_assay.head(5)

,PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,PUBCHEM_ACTIVITY_URL,PUBCHEM_ASSAYDATA_COMMENT,%Activity at 18.75 uM_Mean,%Activity at 18.75 uM_1,...,%Activity at 18.75 uM_3,%Activity at 18.75 uM_4,Value_1,Value_2,Value_3,Value_4,Mean High,STD Deviation High,Mean Low,STD Deviation Low
0,1,85146295.0,44142181.0,C=CS(=O)(=O)N(CC1=CC=CC=C1Br)[C@@H](CC2=CC=CC=...,Inactive,21.0,NaN,NaN,35.14,34.86,...,33.44,36.18,579.04,568.21,591.53,567.48,0,9.38,100,0.43
1,2,14744380.0,3417593.0,COC1=CC(=CC(=C1OC)OC)C(C[N+](=O)[O-])C2=C(C=C3...,Inactive,21.0,NaN,NaN,-37.99,-29.9,...,-38.2,-33.2,1148.02,1330.56,1220.99,1177.05,0,9.38,100,0.43
2,3,99356260.0,285027.0,C1CN2C3=C(C=C(C=C3)Cl)C4=CC=CC=C4C2=N1,Inactive,21.0,NaN,NaN,-58.96,-52.17,...,-52.47,-63.55,1343.67,1479.88,1346.37,1443.7,0,9.38,100,0.43
3,4,22413392.0,4170685.0,COC1=CC=CC(=C1)C2CC3C(=O)C=CC2N3CC4=CC=CC=C4,Active,30.0,NaN,NaN,57.32,56.44,...,56.8,58.6,389.47,380.7,386.25,370.43,0,9.38,100,0.43
4,5,85271027.0,2361547.0,COC1=CC=C(C=C1)/C=C\2/C(=O)N(C(=O)S2)CCCC(=O)N...,Active,30.0,NaN,NaN,98.17,98.1,...,98.66,98.21,23.38,26.82,18.5,22.47,0,9.38,100,0.43


In [47]:
# 활성 판정 분포 확인 (슬라이드의 Active/Inactive 개수와 비교)
df_assay['PUBCHEM_ACTIVITY_OUTCOME'].value_counts()

PUBCHEM_ACTIVITY_OUTCOME
Active      982
Inactive    777
Name: count, dtype: int64

## 2-3. 활성 데이터에 화합물 구조(SMILES) 결합

활성 데이터(`df_assay`)에는 데이터 제공자가 올린 원본 SMILES(`PUBCHEM_EXT_DATASOURCE_SMILES`)만 있고,
**PubChem이 표준화한 SMILES는 없다.**
원본 SMILES는 표기가 제각각이라, 머신러닝에는 CID로 받은 **표준 SMILES**를 쓰는 것이 좋다.
그래서 각 CID의 표준 SMILES를 받아 활성 데이터와 합친다.

In [ ]:
# [0] 활성 데이터에서 유효한 CID만 추출 (결측 제거 후 정수형 변환)
CIDs = df_assay['PUBCHEM_CID'].dropna().astype(int).unique()
target_CIDs = CIDs[:200].tolist()

print('SMILES를 수집할 CID 개수:', len(target_CIDs))

# [1] CID → SMILES 를 담을 빈 딕셔너리 준비
cid_to_smiles = {}

# [2] for loop (반복문) 을 이용해 추출한 CID의 SMILES 요청 (1-2 참조)
for i in range(0, 10):
    cid = target_CIDs[i]

    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/SMILES/TXT'
    
    response = requests.get(url, verify=False)
    cid_to_smiles[cid] = response.text.strip()

    time.sleep(0.2)

print(cid_to_smiles)


SMILES를 수집할 CID 개수: 200


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'pubchem.ncbi.nlm.nih.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'pubchem.ncbi.nlm.nih.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'pubchem.ncbi.nlm.nih.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/l

{44142181: 'C=CS(=O)(=O)N(CC1=CC=CC=C1Br)[C@@H](CC2=CC=CC=C2)CO',
 3417593: 'COC1=CC(=CC(=C1OC)OC)C(C[N+](=O)[O-])C2=C(C=C3N2C=CC=C3)C4=CC=CC=C4',
 285027: 'C1CN2C3=C(C=C(C=C3)Cl)C4=CC=CC=C4C2=N1',
 4170685: 'COC1=CC=CC(=C1)C2CC3C(=O)C=CC2N3CC4=CC=CC=C4',
 2361547: 'COC1=CC=C(C=C1)/C=C\\2/C(=O)N(C(=O)S2)CCCC(=O)NC3=C(C=CC(=C3)Cl)C(=O)O',
 1191331: 'C1CN(CCN1CC2=CC=CC=C2)C3=C(C(=O)N(C3=O)CC4=CC=CC=C4)Cl',
 660935: 'CCOC(=O)C1=C(OC2=C(C1=O)C(=C(C(=C2F)F)F)F)C',
 3715215: 'C=C(C(CC(=O)C1=CC=CC=C1)C2=CC=CC=C2)C(=O)C3=CC=CC=C3',
 24761716: '[2H]C(CCOS(=O)(=O)N)C1=CC=C(C=C1)C(C)(C)C',
 3280153: 'C1=CC=C2C(=C1)N=C(S2)C(=CC3=CC=C(C=C3)C(=O)O)C4=NC5=CC=CC=C5S4'}

In [ ]:
# [3] df_assay에 새 열로 붙이기
df_assay['PUBCHEM_SMILES'] = df_assay['PUBCHEM_CID'].map(cid_to_smiles)
df_assay.head()

,PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,PUBCHEM_ACTIVITY_URL,PUBCHEM_ASSAYDATA_COMMENT,%Activity at 18.75 uM_Mean,%Activity at 18.75 uM_1,...,%Activity at 18.75 uM_4,Value_1,Value_2,Value_3,Value_4,Mean High,STD Deviation High,Mean Low,STD Deviation Low,PUBCHEM_SMILES
0,1,85146295.0,44142181.0,C=CS(=O)(=O)N(CC1=CC=CC=C1Br)[C@@H](CC2=CC=CC=...,Inactive,21.0,NaN,NaN,35.14,34.86,...,36.18,579.04,568.21,591.53,567.48,0,9.38,100,0.43,C=CS(=O)(=O)N(CC1=CC=CC=C1Br)[C@@H](CC2=CC=CC=...
1,2,14744380.0,3417593.0,COC1=CC(=CC(=C1OC)OC)C(C[N+](=O)[O-])C2=C(C=C3...,Inactive,21.0,NaN,NaN,-37.99,-29.9,...,-33.2,1148.02,1330.56,1220.99,1177.05,0,9.38,100,0.43,COC1=CC(=CC(=C1OC)OC)C(C[N+](=O)[O-])C2=C(C=C3...
2,3,99356260.0,285027.0,C1CN2C3=C(C=C(C=C3)Cl)C4=CC=CC=C4C2=N1,Inactive,21.0,NaN,NaN,-58.96,-52.17,...,-63.55,1343.67,1479.88,1346.37,1443.7,0,9.38,100,0.43,C1CN2C3=C(C=C(C=C3)Cl)C4=CC=CC=C4C2=N1
3,4,22413392.0,4170685.0,COC1=CC=CC(=C1)C2CC3C(=O)C=CC2N3CC4=CC=CC=C4,Active,30.0,NaN,NaN,57.32,56.44,...,58.6,389.47,380.7,386.25,370.43,0,9.38,100,0.43,COC1=CC=CC(=C1)C2CC3C(=O)C=CC2N3CC4=CC=CC=C4
4,5,85271027.0,2361547.0,COC1=CC=C(C=C1)/C=C\2/C(=O)N(C(=O)S2)CCCC(=O)N...,Active,30.0,NaN,NaN,98.17,98.1,...,98.21,23.38,26.82,18.5,22.47,0,9.38,100,0.43,COC1=CC=C(C=C1)/C=C\2/C(=O)N(C(=O)S2)CCCC(=O)N...


## 2-4. 데이터 저장

수집한 데이터셋을 CSV 파일로 저장한다.
이 파일은 다음 강의 **`02_preprocessing_rdkit.ipynb`** 에서 전처리에 사용한다.

In [ ]:
out_path = 'hif2a_aid651580_dataset_demo.csv'
df_assay.to_csv(out_path, index=False)
print('저장 완료:', out_path)

# (Colab) 파일 내려받기 - 필요 시 주석 해제
# from google.colab import files
# files.download(out_path)

---
## 정리

이번 실습에서 배운 내용:

| 구분 | 사용한 API 경로 | 얻은 데이터 |
|------|----------------|------------|
| Chemical | `/compound/name|cid/.../property\|synonyms\|description\|PNG` | 분자식·분자량·SMILES·별명·설명·구조 |
| BioAssay | `/assay/aid/{AID}/description\|cids\|CSV` | 어세이 개요·활성 판정·화합물 목록 |

핵심 포인트
- PUG-REST는 **URL만 조합**하면 별도 로그인/키 없이 데이터를 받을 수 있다.
- 출력형식(JSON/CSV/PNG)을 바꿔 상황에 맞게 활용한다.
- 대량 요청 시 **CID를 묶어서** 요청하고 **`time.sleep()`** 으로 서버 부하를 줄인다.
- 수집 결과는 **DataFrame → CSV** 로 저장해 이후 단계로 넘긴다.

**➡️ 다음 강의:** `02_preprocessing_rdkit.ipynb` — 수집한 SMILES를 RDKit으로 전처리하고 분자 특성(descriptor/fingerprint)을 생성한다.